# SAM2 + transparent-background Haystack Pipeline (Google Colab + Gradio 5)

このノートブックは **Jupytext の `.py` を正本** として管理します。`.ipynb` は次のコマンドで生成します。

```powershell
.venv\Scripts\python.exe -m jupytext --to ipynb Sam2_Transparent_Background_Haystack.py
```

実装本体は [gradio_app_sam2_transparent_BG_haystack.py](./gradio_app_sam2_transparent_BG_haystack.py) と [pipelines/](./pipelines/) に集約しています。

**手順:**
1. 依存関係をインストール
2. Google Drive / `PROJECT_ROOT` / checkpoint 配置を解決
3. Haystack 版 Gradio アプリを起動

In [1]:
# ============================================================
# Cell 1: 依存関係インストール
# ============================================================
# -q は使わず、インストールエラーを見える状態にします。
import os
import sys

!{sys.executable} -m pip install gradio==5.9.1
!{sys.executable} -m pip install haystack-ai==2.29.0
!{sys.executable} -m pip install transparent-background
!{sys.executable} -m pip install pymatting
!{sys.executable} -m pip install git+https://github.com/facebookresearch/sam2.git
!{sys.executable} -m pip install opencv-python-headless pillow numpy

print("Install done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.4/320.4 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.2/170.2 kB 20.3 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.3
    Uninstalling MarkupSafe-3.0.3:
      Successfully uninstalled MarkupSafe-3.0.3
  Attempting uninstall: aiofiles
    Found existing installation: aiofiles 24.1.0
    Uninstalling aiofiles-24.1.0:
      Successfully uninstalled aiofiles-24.1.0
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.5

In [2]:
# ============================================================
# Cell 2: Google Drive マウント + プロジェクトパス設定 + チェックポイント配置
# ============================================================
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
COLAB_DRIVE_PROJECT = Path("/content/drive/MyDrive/AI_picasso/Matting-Anything")

if IS_COLAB:
    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.exists():
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
        print("Google Drive mounted at /content/drive")
    else:
        print("Google Drive already mounted")


def detect_project_root() -> Path:
    env_root = os.environ.get("PROJECT_ROOT")
    if env_root:
        return Path(env_root).resolve()
    if IS_COLAB and COLAB_DRIVE_PROJECT.exists():
        return COLAB_DRIVE_PROJECT
    return Path.cwd().resolve()


PROJECT_ROOT = detect_project_root()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
if IS_COLAB and PROJECT_ROOT == COLAB_DRIVE_PROJECT:
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CKPT_ROOT = PROJECT_ROOT / "checkpoints"
SAM2_CKPT_DIR = CKPT_ROOT / "SAM2"
TB_CKPT_DIR = CKPT_ROOT / "transparent_BG"
INPUT_DIR = PROJECT_ROOT / "inputs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
for directory in (SAM2_CKPT_DIR, TB_CKPT_DIR, INPUT_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SAM2_CKPT_PATH = SAM2_CKPT_DIR / "sam2.1_hiera_large.pt"
SAM2_CONFIG_NAME = "configs/sam2.1/sam2.1_hiera_l.yaml"


def fetch_if_missing(path: Path, url: str) -> None:
    if path.exists():
        print(f"Found: {path.relative_to(PROJECT_ROOT)}")
        return
    print(f"Downloading: {path.name}")
    rc = os.system(f'wget -O "{path}" "{url}"')
    if rc != 0 or not path.exists():
        raise RuntimeError(f"Download failed: {url}")
    print(f"Saved: {path.relative_to(PROJECT_ROOT)}")


fetch_if_missing(
    SAM2_CKPT_PATH,
    "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt",
)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SAM2_CKPT_PATH = {SAM2_CKPT_PATH}")
print(f"TB_CKPT_DIR = {TB_CKPT_DIR}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")

Mounted at /content/drive
Google Drive mounted at /content/drive
Found: checkpoints/SAM2/sam2.1_hiera_large.pt
PROJECT_ROOT = /content/drive/MyDrive/AI_picasso/Matting-Anything
SAM2_CKPT_PATH = /content/drive/MyDrive/AI_picasso/Matting-Anything/checkpoints/SAM2/sam2.1_hiera_large.pt
TB_CKPT_DIR = /content/drive/MyDrive/AI_picasso/Matting-Anything/checkpoints/transparent_BG
OUTPUT_DIR = /content/drive/MyDrive/AI_picasso/Matting-Anything/outputs


In [ ]:
# ============================================================
# Cell 3: Haystack 版 Gradio アプリ起動
# ============================================================
APP_PATH = PROJECT_ROOT / "gradio_app_sam2_transparent_BG_haystack.py"
assert APP_PATH.exists(), f"アプリが見つかりません: {APP_PATH}"

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["SAM2_CKPT_PATH"] = str(SAM2_CKPT_PATH)
os.environ["SAM2_CONFIG_NAME"] = SAM2_CONFIG_NAME

IS_COLAB = "google.colab" in sys.modules
SHARE_FLAG = "--share" if IS_COLAB else ""

!{sys.executable} "{APP_PATH}" {SHARE_FLAG}

* Running on local URL:  http://0.0.0.0:7862
* Running on public URL: https://de2b9ec5c7a4b82243.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Traceback (most recent call last):
  File "/content/drive/MyDrive/AI_picasso/Matting-Anything/gradio_app_sam2_transparent_BG_haystack.py", line 225, in predict_masks
    raise gr.Error("画像をクリックして Point または Box を指定してください。")
gradio.exceptions.Error: '画像をクリックして Point または Box を指定してください。'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 322, in call_